# Bundestagswahl 2021–2025: vote shares and selected migration

Party bar heights show each election's full percentage of valid second votes on a common scale. Ribbons retain the selected migration connections; each endpoint uses the corresponding election's denominator. Unused bar space means **not represented in this flow dataset**, not necessarily voters who stayed with the party.

Non-voters occupy a separate band and use eligible voters as their denominator. They are not part of the party totals. All parties, including FDP 2025 and an Other parties remainder, are included even when no selected flow reaches them.

The migration estimates come from `data/bundestagswahl_2025_voter_migration_sankey_plot_ready.csv`; complete election totals come from `data/bundestagswahl_election_results.json` (official results, including the 2024 partial Berlin repeat for 2021). Geometry uses unrounded percentages; labels are rounded for readability. The export cell writes the responsive HTML page to `p00/output`.

> **Method:** The migration data consists of selected model-based estimates from tagesschau / infratest dimap, not a complete gross migration matrix or exact ballot tracking. The chart does not infer missing flows or voter retention. Ribbons can taper because denominators differ between elections and between parties and non-voters.


In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go


def find_project_root(start: Path = Path.cwd()) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if candidate.name == "p00":
            return candidate
    raise RuntimeError(f"Could not find a parent folder named 'p00' from {start.resolve()}")


project_root = find_project_root()
csv_path = project_root / "data" / "bundestagswahl_2025_voter_migration_sankey_plot_ready.csv"
if not csv_path.is_file():
    raise FileNotFoundError(f"Migration CSV not found: {csv_path}")

data = pd.read_csv(csv_path, encoding="utf-8")
required_columns = {
    "record_type",
    "source_id",
    "source_label",
    "source_party_long",
    "source_year",
    "source_vote_share_pct",
    "source_share_basis",
    "source_votes_total",
    "source_sort_order",
    "source_x",
    "source_color_hex",
    "target_id",
    "target_label",
    "target_party_long",
    "target_year",
    "target_vote_share_pct",
    "target_share_basis",
    "target_votes_total",
    "target_sort_order",
    "target_x",
    "target_color_hex",
    "value_voters",
    "flow_label",
    "flow_color_rgba",
    "estimate_type",
    "migration_source_url",
    "plot_note",
}
missing_columns = required_columns - set(data.columns)
if missing_columns:
    raise ValueError(f"Migration CSV is missing required columns: {sorted(missing_columns)}")

data = data.loc[data["record_type"].eq("flow")].copy()
data = data.dropna(subset=["source_id", "target_id", "source_label", "target_label", "value_voters"])
data["value_voters"] = pd.to_numeric(data["value_voters"], errors="raise")
if data.empty or (data["value_voters"] <= 0).any():
    raise ValueError("Migration CSV must contain positive voter estimates.")
if set(data["source_year"].unique()) != {2021} or set(data["target_year"].unique()) != {2025}:
    raise ValueError("Expected migration flows from 2021 sources to 2025 targets.")

print(f"Loaded {len(data)} estimated flows ({data['value_voters'].sum():,.0f} voters represented).")

In [ ]:
import json
from html import escape

import numpy as np

share_basis_labels = {
    "valid_second_votes": "Valid second votes",
    "eligible_voters": "Eligible voters",
}
elections = json.loads(
    (project_root / "data" / "bundestagswahl_election_results.json").read_text(encoding="utf-8")
)["elections"]

# Build complete election bars independently of the incomplete migration matrix.
party_styles = [
    ("union", "Union", "CDU/CSU", "#D4D8DE"),
    ("spd", "SPD", "Sozialdemokratische Partei Deutschlands", "#E31B23"),
    ("gruene", "Grüne", "Bündnis 90/Die Grünen", "#64B914"),
    ("fdp", "FDP", "Freie Demokratische Partei", "#FFCC00"),
    ("afd", "AfD", "Alternative für Deutschland", "#009EE0"),
    ("linke", "Linke", "Die Linke", "#BE3075"),
    ("bsw", "BSW", "Bündnis Sahra Wagenknecht", "#AD4577"),
    ("other", "Other parties", "All remaining parties", "#8294AB"),
    ("nichtwaehler", "Nichtwähler", "Non-voters", "#808080"),
]


def visible_node_color(color_hex: str) -> str:
    return "#D4D8DE" if color_hex.lower() in {"#000000", "#111111"} else color_hex


def rgba_from_hex(color_hex: str, alpha: float = 0.38) -> str:
    color_hex = color_hex.lstrip("#")
    red, green, blue = (int(color_hex[index:index + 2], 16) for index in (0, 2, 4))
    return f"rgba({red},{green},{blue},{alpha})"


def build_nodes(year: int) -> pd.DataFrame:
    result = elections[str(year)]
    totals = dict(result["party_votes"])
    totals["other"] = result["valid_second_votes"] - sum(totals.values())
    totals["nichtwaehler"] = result["eligible_voters"] - result["voters"]
    if min(totals.values()) < 0:
        raise ValueError(f"Invalid election totals for {year}")
    rows = []
    top = 0.0
    for key, party, party_long, color in party_styles:
        # A zero result has no area and no migration endpoint (BSW in 2021).
        if totals[key] == 0:
            continue
        basis = "eligible_voters" if key == "nichtwaehler" else "valid_second_votes"
        denominator = result[basis]
        share = 100 * totals[key] / denominator
        if key == "nichtwaehler":
            top = 128.0  # Separate band; never add this to the 100% party total.
        share_display = f"{share:.3f}" if key == "bsw" else f"{share:.1f}"
        rows.append(dict(
            node_id=f"{key}_{year}", party=party, party_long=party_long, year=year,
            label=f"{party} {year} · {share_display}%", share_display=share_display,
            vote_share_pct=share, share_basis=basis, votes_total=totals[key],
            denominator=denominator, color_hex=color, top=top, bottom=top + share,
        ))
        top += share + 2.0
    return pd.DataFrame(rows)


source_nodes = build_nodes(2021)
target_nodes = build_nodes(2025)
nodes = pd.concat([source_nodes, target_nodes], ignore_index=True)
node_lookup = nodes.set_index("node_id").to_dict("index")

# Each endpoint has its own percentage width. Renormalizing the selected flows
# to fill the bars would misrepresent the incomplete migration estimates.
flows = data.copy().reset_index(drop=True)
for side in ("source", "target"):
    unknown = set(flows[f"{side}_id"]) - set(node_lookup)
    if unknown:
        raise ValueError(f"Unknown {side} migration nodes: {sorted(unknown)}")
    flows[f"{side}_pct"] = [
        100 * row.value_voters / node_lookup[getattr(row, f"{side}_id")]["denominator"]
        for row in flows.itertuples(index=False)
    ]
    for node_id, group in flows.groupby(f"{side}_id"):
        node = node_lookup[node_id]
        if group["value_voters"].sum() > node["votes_total"]:
            raise ValueError(f"Selected migration estimates exceed the total for {node_id}")
        opposite = "target" if side == "source" else "source"
        order = sorted(group.index, key=lambda i: node_lookup[flows.at[i, f"{opposite}_id"]]["top"])
        cursor = node["top"]
        for index in order:
            flows.at[index, f"{side}_top"] = cursor
            cursor += flows.at[index, f"{side}_pct"]
            flows.at[index, f"{side}_bottom"] = cursor

fig = go.Figure()
bar_width = 0.024
left_edge, right_edge = bar_width, 1.0 - bar_width
t = np.linspace(0, 1, 64)
# Sample a cubic Bezier with horizontal tangents at each endpoint.
curve_x = ((1-t)**3 * left_edge + 3*(1-t)**2*t * 0.40
           + 3*(1-t)*t**2 * 0.60 + t**3 * right_edge)
blend = 3*t**2 - 2*t**3

for row in flows.itertuples(index=False):
    source = node_lookup[row.source_id]
    target = node_lookup[row.target_id]
    upper = row.source_top + blend * (row.target_top - row.source_top)
    lower = row.source_bottom + blend * (row.target_bottom - row.source_bottom)
    hover = (
        f"<b>{escape(source['party'])} 2021 → {escape(target['party'])} 2025</b>"
        f"<br>{row.value_voters:,.0f} estimated voters"
        f"<br>2021 endpoint: {row.source_pct:.3f}% of {share_basis_labels[source['share_basis']].lower()}"
        f"<br>2025 endpoint: {row.target_pct:.3f}% of {share_basis_labels[target['share_basis']].lower()}"
        "<br>Selected migration estimate; incomplete coverage"
    )
    fig.add_trace(go.Scatter(
        x=np.concatenate([curve_x, curve_x[::-1], curve_x[:1]]).tolist(),
        y=np.concatenate([upper, lower[::-1], upper[:1]]).tolist(),
        mode="lines", line=dict(width=0), fill="toself",
        fillcolor=rgba_from_hex(source["color_hex"], 0.34),
        hoveron="fills", text=hover, hoverinfo="text", showlegend=False,
        name=f"{source['party']} → {target['party']}",
        meta=dict(kind="flow", source=row.source_id, target=row.target_id),
    ))

for side, side_nodes, x0, x1 in (
    ("source", source_nodes, 0.0, bar_width),
    ("target", target_nodes, 1.0 - bar_width, 1.0),
):
    for row in side_nodes.itertuples(index=False):
        selected = float(flows.loc[flows[f"{side}_id"].eq(row.node_id), "value_voters"].sum())
        used_pct = 100 * selected / row.denominator
        basis = share_basis_labels[row.share_basis].lower()
        hover = (
            f"<b>{escape(row.party)} {row.year}: {row.share_display}%</b>"
            f"<br>{row.votes_total:,} {'non-voters' if row.share_basis == 'eligible_voters' else 'votes'}"
            f"<br>Basis: {row.denominator:,} {basis}"
            f"<br>Selected flows: {used_pct:.3f} percentage points"
            f"<br>Not represented: {row.vote_share_pct - used_pct:.3f} percentage points"
        )
        fig.add_trace(go.Scatter(
            x=[x0, x1, x1, x0, x0], y=[row.top, row.top, row.bottom, row.bottom, row.top],
            mode="lines", line=dict(color=row.color_hex, width=1),
            fill="toself", fillcolor=rgba_from_hex(row.color_hex, 0.20),
            hoveron="fills", text=hover, hoverinfo="text", showlegend=False,
            name=row.label, meta=dict(kind="node", node_id=row.node_id),
        ))
        if used_pct:
            fig.add_shape(
                type="rect", x0=x0, x1=x1, y0=row.top, y1=row.top + used_pct,
                fillcolor=row.color_hex, line=dict(width=0), layer="above",
                name=f"used-{row.node_id}",
            )
        fig.add_annotation(
            x=x0 - 0.016 if side == "source" else x1 + 0.016,
            y=(row.top + row.bottom) / 2,
            text=f"{escape(row.party)} <b>{row.share_display}%</b>",
            name=f"node-{row.node_id}",
            xanchor="right" if side == "source" else "left",
            showarrow=False, font=dict(size=12),
        )

fig.update_layout(
    template="plotly_dark", paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    font=dict(family="system-ui, sans-serif", size=13, color="#f5f7fa"),
    width=None, height=1040, autosize=True, margin=dict(l=8, r=8, t=65, b=26),
    xaxis=dict(range=[-0.29, 1.29], visible=False, fixedrange=True),
    yaxis=dict(range=[154, -1], visible=False, fixedrange=True),
    dragmode=False, hovermode="closest", hoverlabel=dict(namelength=-1),
)
for x, year in ((0, 2021), (1, 2025)):
    fig.add_annotation(x=x, y=1.055, yref="paper", text=f"<b>{year} VOTE SHARE</b>", name=f"year-{year}", showarrow=False)
    fig.add_annotation(x=x, y=1.029, yref="paper", text="Valid second votes · total 100%", name=f"basis-{year}", showarrow=False, font=dict(size=11, color="#aeb8c4"))
fig.add_annotation(x=0.5, y=121.5, text="NON-VOTERS · % of eligible voters · separate from party totals", name="non-voters-basis", showarrow=False, font=dict(size=11, color="#aeb8c4"))
fig.add_shape(type="line", x0=-0.1, x1=1.1, y0=118, y1=118, line=dict(color="#30363d", width=1, dash="dot"))

# Share the same guidance between the notebook output and the exported page.
from IPython.display import HTML, display

migration_urls = data["migration_source_url"].dropna().astype(str).drop_duplicates().tolist()
migration_url = next((url for url in migration_urls if "/charts/" not in url), migration_urls[0])
election_url_2021 = elections["2021"]["source_url"]
election_url_2025 = elections["2025"]["source_url"]
wahlrecht_url = data["wahlrecht_summary_url"].dropna().astype(str).iloc[0]
plot_note = (
    "Bar heights show full election percentages: parties use valid second votes; "
    "non-voters use eligible voters. Ribbons show selected model-based migration estimates, "
    "not exact ballot tracking or a complete migration matrix. Each endpoint uses its "
    "election's denominator, so ribbons can taper. Dim bar space means voters absent "
    "from the flow data, not necessarily voters who stayed with a party."
)
chart_guidance_html = f"""<div class="chart-guidance">
  <p class="method-note" style="border-left:3px solid #d4a72c;padding-left:0.8rem;line-height:1.5"><strong>Method:</strong> {escape(plot_note)}</p>
  <p class="sources" style="line-height:1.7"><strong>Sources:</strong>
    <a href="{escape(migration_url, quote=True)}" target="_blank" rel="noopener noreferrer">tagesschau / infratest dimap migration estimates</a>;
    official <a href="{escape(election_url_2021, quote=True)}" target="_blank" rel="noopener noreferrer">2021</a> and
    <a href="{escape(election_url_2025, quote=True)}" target="_blank" rel="noopener noreferrer">2025 election results</a>;
    <a href="{escape(wahlrecht_url, quote=True)}" target="_blank" rel="noopener noreferrer">Wahlrecht.de</a> for election-results context.
    The 2021 results include the partial Berlin repeat election in 2024.
  </p>
</div>
"""
display(HTML(chart_guidance_html))
fig.show()


In [ ]:
from html import escape

project_root = find_project_root()
output_dir = project_root / "output"
output_dir.mkdir(exist_ok=True)
output_file = output_dir / "20261008_bundestagswahl_sankey.html"

# Embed the shared responsive layout and drag controls in the exported HTML.
responsive_script = (project_root / "assets" / "bundestag_sankey.js").read_text(encoding="utf-8")

plot_html = fig.to_html(
    include_plotlyjs="cdn",
    full_html=False,
    post_script=responsive_script.strip(),
    config={"responsive": True, "displaylogo": False, "scrollZoom": False},
)

html_document = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <meta name="theme-color" content="#0d1117">
  <title>Bundestag vote shares and migration 2021–2025 | phey.app</title>
  <style>
    * {{ box-sizing: border-box; }}
    html, body {{ margin: 0; min-height: 100%; background: #0d1117; color: #f5f7fa; }}
    body {{ font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; }}
    .plot-page {{ min-height: 100vh; padding: clamp(0.75rem, 2vw, 1.5rem); }}
    .plot-wrap {{ width: 100%; margin: 0 auto; }}
    .back-link {{ display: inline-flex; margin-bottom: 1rem; color: #c7d0dd; text-decoration: none; }}
    .back-link:hover {{ color: #ffffff; }}
    h1 {{ margin: 0; font-size: clamp(1.55rem, 3vw, 2.25rem); font-weight: 650; }}
    .subtitle {{ max-width: 75rem; margin: 0.45rem 0 0; color: #c7d0dd; line-height: 1.5; }}
    .method-note {{ max-width: 90rem; margin: 0.75rem 0 0.25rem; padding-left: 0.8rem; border-left: 3px solid #d4a72c; color: #aeb8c4; line-height: 1.5; }}
    .sources {{ margin: 0.75rem 0 1.5rem; color: #8b949e; font-size: 0.9rem; line-height: 1.7; }}
    .sources a {{ color: #8ecae6; }}
    .sankey-plot {{ min-width: 0; width: 100%; }}
    .sankey-plot .plotly-graph-div {{ width: 100%; min-width: 0; position: relative; }}
    .chart-controls {{ display: flex; align-items: center; justify-content: space-between; gap: 0.75rem; margin: 0.6rem 0; }}
    .chart-controls p {{ margin: 0; color: #aeb8c4; font-size: 0.85rem; line-height: 1.4; }}
    #reset-layout {{ flex-shrink: 0; min-height: 44px; padding: 0.5rem 0.75rem; background: #161b22; color: #f5f7fa; border: 1px solid #59636e; border-radius: 6px; cursor: pointer; }}
    #reset-layout:focus-visible {{ outline: 2px solid #8ecae6; outline-offset: 3px; }}
    #node-selection {{ display: block; margin: 0.5rem 0; color: #c7d0dd; font-size: 0.85rem; line-height: 1.5; }}
    #node-selection[hidden] {{ display: none; }}
    .node-handles {{ position: absolute; inset: 0; pointer-events: none; z-index: 5; }}
    .node-handle {{ position: absolute; margin: 0; padding: 0; border: 0; background: transparent; pointer-events: auto; touch-action: none; cursor: grab; }}
    .node-handle.is-dragging {{ cursor: grabbing; }}
    .node-handle:focus-visible {{ outline: 2px solid #8ecae6; outline-offset: 2px; }}
    .guidance-toggle {{ margin-top: 0.75rem; }}
    .guidance-toggle summary {{ width: fit-content; min-height: 44px; padding: 0.65rem 0; cursor: pointer; color: #c7d0dd; font-size: 0.9rem; }}
    .guidance-toggle summary:focus-visible {{ outline: 2px solid #8ecae6; outline-offset: 3px; }}
    .back-link {{ min-height: 44px; align-items: center; }}
    @media (max-width: 640px) {{
      .plot-page {{ padding: 0.65rem; }}
      .back-link {{ margin-bottom: 0.35rem; }}
      h1 {{ font-size: 1.35rem; line-height: 1.25; }}
      .subtitle, .method-note, .sources {{ font-size: 0.85rem; }}
      .sources {{ overflow-wrap: anywhere; }}
      .guidance-toggle {{ margin: 0.3rem 0 0.6rem; }}
      .sankey-plot .modebar {{ display: none; }}
    }}
  </style>
</head>
<body>
  <main class="plot-page">
    <div class="plot-wrap">
      <a class="back-link" href="../index.html">← Back to phey.app</a>
      <header>
        <h1>Bundestag vote shares &amp; migration, 2021 to 2025</h1>
        <p class="subtitle">Bar heights show the full election percentages in 2021 and 2025. Colored ribbons connect the selected migration estimates; dim bar space marks the portion absent from the flow data.</p>
        <details class="guidance-toggle" open>
          <summary>Method &amp; sources</summary>
          {chart_guidance_html}
        </details>
      </header>
      <div class="chart-controls">
        <p id="drag-hint">Drag a bar or its label to move the connected ribbons. Tap for details.</p>
        <button id="reset-layout" type="button">Reset layout</button>
      </div>
      <output id="node-selection" aria-live="polite" hidden></output>
      <div class="sankey-plot" aria-describedby="drag-hint" role="region" aria-label="Vote shares and selected voter migration for 2021 and 2025" tabindex="0">{plot_html}</div>
    </div>
  </main>
</body>
</html>
"""

output_file.write_text(html_document, encoding="utf-8")
print(f"Exported Bundestag Sankey to: {output_file.resolve()}")